In [4]:
"""
E-Commerce Churn Dataset — Full Preprocessing Pipeline
Steps: Outlier handling (IQR capping + unstable-row removal) ->
       Skewness check + log transform ->
       Feature engineering (encoding + scaling) via ColumnTransformer ->
       Train/test split + SMOTE demo (validation only, not saved) ->
       Save single fully preprocessed CSV
"""

import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder, FunctionTransformer
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

pd.set_option('display.width', 140)

# ============================================================
# STEP 0: LOAD DATA
# ============================================================
df = pd.read_csv('/Users/meetnakrani/Library/Mobile Documents/com~apple~CloudDocs/Customer-Retention-intelligence-Platform/DataSet/cleaned/E_Comm_Cleaned_Final.csv')
print(f"Original shape: {df.shape}")

df = df.drop(columns=['CustomerID'])  # identifier, no predictive value

TARGET = 'Churn'

# ============================================================
# STEP 1: OUTLIER HANDLING
#   1a. Remove genuinely unstable/extreme rows (beyond 3x IQR, rare count)
#   1b. Cap remaining outliers using standard 1.5x IQR
# ============================================================

# columns where a handful of extreme rows are true anomalies -> delete row
unstable_cols = ['Tenure', 'NumberOfAddress', 'DaySinceLastOrder']

rows_before = df.shape[0]
mask_keep = pd.Series(True, index=df.index)
for col in unstable_cols:
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 3 * IQR, Q3 + 3 * IQR
    mask_keep &= df[col].between(lower, upper)

df = df[mask_keep].reset_index(drop=True)
print(f"Removed {rows_before - df.shape[0]} unstable outlier rows -> new shape: {df.shape}")

# columns to cap using standard IQR (kept, since these are real skewed behavior, not errors)
outlier_cap_cols = ['Tenure', 'WarehouseToHome', 'HourSpendOnApp', 'NumberOfDeviceRegistered',
                    'NumberOfAddress', 'OrderAmountHikeFromlastYear', 'CouponUsed', 'OrderCount',
                    'DaySinceLastOrder', 'CashbackAmount']

for col in outlier_cap_cols:
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    df[col] = df[col].clip(lower, upper)

print("IQR capping applied to:", outlier_cap_cols)

# ============================================================
# STEP 2: SKEWNESS CHECK
# ============================================================
numeric_cols = ['Tenure', 'WarehouseToHome', 'HourSpendOnApp', 'NumberOfDeviceRegistered',
                'NumberOfAddress', 'OrderAmountHikeFromlastYear', 'CouponUsed', 'OrderCount',
                'DaySinceLastOrder', 'CashbackAmount', 'CityTier', 'SatisfactionScore']

skew_vals = df[numeric_cols].skew().sort_values(ascending=False)
print("\nSkewness after capping:")
print(skew_vals)

SKEW_THRESHOLD = 0.5
skewed_cols = skew_vals[abs(skew_vals) > SKEW_THRESHOLD].index.tolist()
non_skewed_numeric_cols = [c for c in numeric_cols if c not in skewed_cols]

print(f"\nSkewed columns (log1p transform): {skewed_cols}")
print(f"Non-skewed numeric columns (scale only): {non_skewed_numeric_cols}")

# ============================================================
# STEP 3: COLUMN GROUPING FOR FEATURE ENGINEERING
# ============================================================
binary_cat_cols = ['PreferredLoginDevice', 'Gender']          # 2 categories -> ordinal (0/1)
nominal_cat_cols = ['PreferredPaymentMode', 'PreferedOrderCat', 'MaritalStatus']  # >2 categories -> one-hot
already_binary_numeric = ['Complain']  # passthrough, already 0/1

# ============================================================
# STEP 4: BUILD PIPELINES + COLUMNTRANSFORMER
# ============================================================
skewed_pipeline = Pipeline(steps=[
    ('log_transform', FunctionTransformer(np.log1p, validate=False)),
    ('scaler', StandardScaler())
])

numeric_pipeline = Pipeline(steps=[
    ('scaler', StandardScaler())
])

binary_pipeline = Pipeline(steps=[
    ('ordinal_encoder', OrdinalEncoder())
])

onehot_pipeline = Pipeline(steps=[
    ('onehot_encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('skewed', skewed_pipeline, skewed_cols),
    ('numeric', numeric_pipeline, non_skewed_numeric_cols),
    ('binary_cat', binary_pipeline, binary_cat_cols),
    ('nominal_cat', onehot_pipeline, nominal_cat_cols),
    ('binary_num', 'passthrough', already_binary_numeric),
], remainder='drop')

X = df.drop(columns=[TARGET])
y = df[TARGET]

X_transformed = preprocessor.fit_transform(X)

# ============================================================
# STEP 4.5: SAVE THE FITTED PREPROCESSOR (needed for FastAPI)
#   This lets the API transform raw user input the exact same
#   way training data was transformed — critical for correct predictions.
# ============================================================
import joblib

joblib.dump(preprocessor, 'preprocessor.pkl')
print("Saved fitted preprocessor -> preprocessor.pkl")

# Also save the RAW input column order (before transformation)
# This is what the API will expect from the user-facing form
raw_input_columns = list(X.columns)
joblib.dump(raw_input_columns, 'raw_input_columns.pkl')
print(f"Saved raw input column order -> raw_input_columns.pkl")
print(f"Raw columns ({len(raw_input_columns)}): {raw_input_columns}")

# ---- reconstruct proper column names ----
onehot_feature_names = preprocessor.named_transformers_['nominal_cat'] \
    .named_steps['onehot_encoder'].get_feature_names_out(nominal_cat_cols)

final_columns = (
    skewed_cols +
    non_skewed_numeric_cols +
    binary_cat_cols +
    list(onehot_feature_names) +
    already_binary_numeric
)

X_final = pd.DataFrame(X_transformed, columns=final_columns, index=X.index)
preprocessed_df = pd.concat([X_final, y.reset_index(drop=True)], axis=1)

print(f"\nFinal preprocessed shape: {preprocessed_df.shape}")
print(preprocessed_df.head())

# ============================================================
# STEP 5: TRAIN/TEST SPLIT FIRST, THEN SMOTE ON TRAINING SET ONLY
#   This avoids data leakage — test set stays real & untouched,
#   so evaluation metrics reflect true generalization.
# ============================================================
X_train, X_test, y_train, y_test = train_test_split(
    X_final, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")
print(f"Train class balance (before SMOTE): {y_train.value_counts().to_dict()}")
print(f"Test class balance (kept real, untouched): {y_test.value_counts().to_dict()}")

# Apply SMOTE ONLY on training data
X_train_bal, y_train_bal = SMOTE(random_state=42).fit_resample(X_train, y_train)
print(f"\nTrain shape after SMOTE: {X_train_bal.shape}")
print(f"Train class balance (after SMOTE): {y_train_bal.value_counts().to_dict()}")

# ============================================================
# STEP 6: SAVE FINAL PREPROCESSED CSV (full dataset, unsplit — for reference/EDA)
# ============================================================
preprocessed_df.to_csv('E_Comm_Preprocessed.csv', index=False)
print("\nSaved full preprocessed dataset -> E_Comm_Preprocessed.csv")

# ============================================================
# STEP 7: SAVE TRAIN (BALANCED) AND TEST (UNTOUCHED) SEPARATELY
#   These are the files you should actually train/evaluate models on.
# ============================================================
train_df = pd.concat([X_train_bal.reset_index(drop=True), y_train_bal.reset_index(drop=True)], axis=1)
test_df = pd.concat([X_test.reset_index(drop=True), y_test.reset_index(drop=True)], axis=1)

train_df.to_csv('E_Comm_Train_Balanced.csv', index=False)
test_df.to_csv('E_Comm_Test_Real.csv', index=False)

print(f"\nSaved training set (SMOTE-balanced) -> E_Comm_Train_Balanced.csv, shape: {train_df.shape}")
print(f"Saved test set (real, untouched)     -> E_Comm_Test_Real.csv, shape: {test_df.shape}")

Original shape: (5630, 20)
Removed 9 unstable outlier rows -> new shape: (5621, 19)
IQR capping applied to: ['Tenure', 'WarehouseToHome', 'HourSpendOnApp', 'NumberOfDeviceRegistered', 'NumberOfAddress', 'OrderAmountHikeFromlastYear', 'CouponUsed', 'OrderCount', 'DaySinceLastOrder', 'CashbackAmount']

Skewness after capping:
OrderCount                     1.147304
NumberOfAddress                0.963205
WarehouseToHome                0.944700
CashbackAmount                 0.892716
OrderAmountHikeFromlastYear    0.823734
DaySinceLastOrder              0.814464
CityTier                       0.735370
Tenure                         0.689402
CouponUsed                     0.531525
HourSpendOnApp                -0.032665
SatisfactionScore             -0.142873
NumberOfDeviceRegistered      -0.304150
dtype: float64

Skewed columns (log1p transform): ['OrderCount', 'NumberOfAddress', 'WarehouseToHome', 'CashbackAmount', 'OrderAmountHikeFromlastYear', 'DaySinceLastOrder', 'CityTier', 'Tenure',